In [ ]:
# EncoderGuiSmoke.ipynb
# Phase 7G+ rotary-encoder HDMI GUI control with live-apply.
# Single-cell notebook. Operates the AudioLabOverlay through the
# compact-v2 GUI; raw register dumps and ipywidgets sub-UIs have been
# removed on purpose.
#
# Operation:
#   Encoder1 rotate     -> select effect (GUI EFFECTS order)
#   Encoder1 short      -> toggle ON/OFF of the selected effect + live apply
#   Encoder1 long       -> Safe Bypass (and second long un-bypasses)
#   Encoder2 rotate     -> select parameter (or model when in model_select_mode)
#   Encoder2 short      -> toggle model_select_mode for PEDAL / AMP / CAB
#                          (or edit_mode for other effects)
#   Encoder3 rotate     -> change focused parameter value + throttled live apply
#   Encoder3 short      -> force-apply current AppState to the overlay
#   Encoder3 long       -> reset focused parameter to its GUI default
#
# RAT (Distortion pedal-mask bit 2) is intentionally EXCLUDED from
# encoder-driven control. The Clash stage and notebook mirror API are
# untouched; only the encoder runtime skips RAT.
#
# Smoke baseline:
#   axi_encoder_input ip_dict key : enc_in_0/s_axi
#   EXPECTED_VERSION              : 0x00070001
#   CONFIG_DEFAULT                : 0x00010105
#   VTC GEN_ACTSZ                 : 0x02580320  (SVGA 800x600 @ 40 MHz)
#
# The loop is dirty-flag driven: at idle proc_cpu stays low; render and
# framebuffer write only run when the AppState a rendered frame depends
# on actually changes, with a cap of MAX_RENDER_FPS during continuous
# rotation. Resource usage is printed every STATUS_INTERVAL_S regardless
# of render activity.

import os
import sys
import time

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "GUI")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.encoder_input import (
    EncoderInput,
    EXPECTED_VERSION,
    CONFIG_DEFAULT,
)
from audio_lab_pynq.encoder_ui import EncoderUiController
from audio_lab_pynq.encoder_effect_apply import EncoderEffectApplier
from audio_lab_pynq.hdmi_backend import (
    AudioLabHdmiBackend,
    DEFAULT_WIDTH,
    DEFAULT_HEIGHT,
)
from audio_lab_pynq.hdmi_state.resource_sampler import ResourceSampler
from GUI.compact_v2.state import AppState
from GUI.pynq_multi_fx_gui import render_frame_800x480_compact_v2


def _find_encoder_ip_name(overlay):
    ip_dict = getattr(overlay, "ip_dict", {})
    for name in ("enc_in_0/s_axi", "enc_in_0",
                 "axi_encoder_input_0", "axi_encoder_input"):
        if name in ip_dict:
            return name
    for name in ip_dict:
        lower = name.lower()
        if "encoder" in lower or lower.startswith("enc_in"):
            return name
    raise RuntimeError("encoder IP not found in overlay.ip_dict")


# --- Tunables -------------------------------------------------------------
GUI_SECONDS        = 600
POLL_HZ_ACTIVE     = 10        # 100 ms while events are arriving
POLL_HZ_IDLE       = 4         # 250 ms after IDLE_THRESHOLD_S of silence
IDLE_THRESHOLD_S   = 1.0
MAX_RENDER_FPS     = 5
STATUS_INTERVAL_S  = 2.0
LIVE_APPLY         = True      # encoder3 rotate -> overlay (throttled)
APPLY_INTERVAL_MS  = 100
VALUE_STEP         = 5.0
SKIP_RAT           = True      # encoder cycling/live apply skip RAT pedal
NO_AUDIO_APPLY     = False     # set True to keep GUI but skip overlay writes
RUN_ENCODER_GUI    = True


def stop_encoder_gui():
    """Call from a later cell or signal to break out of the loop cleanly."""
    global RUN_ENCODER_GUI
    RUN_ENCODER_GUI = False


# --- Overlay attach (once per kernel) -------------------------------------
if "ovl" not in globals():
    ovl = AudioLabOverlay()

# --- Smoke baseline (cheap) ----------------------------------------------
from pynq import MMIO as _MMIO
_vtc_desc = ovl.ip_dict["v_tc_hdmi"]
_vtc_actsz = int(_MMIO(int(_vtc_desc["phys_addr"]),
                       int(_vtc_desc["addr_range"])).read(0x60))
assert _vtc_actsz == 0x02580320, ("VTC GEN_ACTSZ = 0x%08X (expected "
                                  "0x02580320 SVGA 800x600 @ 40 MHz)"
                                  % _vtc_actsz)

# --- Encoder + GUI wiring -------------------------------------------------
enc = EncoderInput.from_overlay(ovl, ip_name=_find_encoder_ip_name(ovl))
_version = enc.read_version()
assert _version == EXPECTED_VERSION, (
    "encoder VERSION = 0x%08X expected 0x%08X" % (_version, EXPECTED_VERSION))
enc.configure(clear_on_read=True)
_cfg = enc.read_config()
assert (_cfg & 0xFFFF) == (CONFIG_DEFAULT & 0xFFFF), (
    "encoder CONFIG = 0x%08X expected ~0x%08X" % (_cfg, CONFIG_DEFAULT))

state = AppState()
state.live_apply = bool(LIVE_APPLY)
state.apply_interval_ms = int(APPLY_INTERVAL_MS)

applier_overlay = None if NO_AUDIO_APPLY else ovl
applier = EncoderEffectApplier(
    applier_overlay,
    apply_interval_s=max(0.001, float(APPLY_INTERVAL_MS) / 1000.0),
    dry_run=bool(NO_AUDIO_APPLY),
    skip_rat=bool(SKIP_RAT),
)
controller = EncoderUiController(
    state, applier=applier, live_apply=bool(LIVE_APPLY),
    skip_rat=bool(SKIP_RAT), value_step=float(VALUE_STEP),
    apply_on_value_change=False,
)
backend = AudioLabHdmiBackend(ovl, width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT)
backend.start()

print("Encoder GUI ready. live_apply=%s skip_rat=%s apply_interval_ms=%d"
      % (LIVE_APPLY, SKIP_RAT, APPLY_INTERVAL_MS))
print("Encoder1 rotate=effect, short=on/off, long=safe-bypass")
print("Encoder2 rotate=param/model, short=model/edit, long=reset model mode")
print("Encoder3 rotate=value (live apply), short=force apply, long=reset knob")
print("RAT pedal model is excluded from encoder control (skip_rat=%s)"
      % SKIP_RAT)


def _render_signature(s):
    knobs = ()
    akv = getattr(s, "all_knob_values", None)
    if isinstance(akv, dict):
        knobs = tuple((k, tuple(v)) for k, v in akv.items())
    return (
        getattr(s, "selected_effect", None),
        getattr(s, "selected_knob", None),
        bool(getattr(s, "value_dirty", False)),
        bool(getattr(s, "apply_pending", False)),
        bool(getattr(s, "edit_mode", False)),
        bool(getattr(s, "model_select_mode", False)),
        getattr(s, "dist_model_idx", None),
        getattr(s, "amp_model_idx", None),
        getattr(s, "cab_model_idx", None),
        bool(getattr(s, "last_apply_ok", True)),
        str(getattr(s, "last_apply_message", "")),
        id(getattr(s, "last_encoder_event", None)),
        tuple(bool(v) for v in (getattr(s, "effect_on", []) or [])),
        knobs,
    )


def _fmt_pct(v):
    return ("%5.1f%%" % v) if v is not None else "  n/a"


def _fmt_temp(v):
    return ("%4.1fC" % v) if v is not None else "  n/a"


sampler = ResourceSampler()
sampler.sample()  # bootstrap

poll_period_active = 1.0 / float(POLL_HZ_ACTIVE)
poll_period_idle   = 1.0 / float(POLL_HZ_IDLE)
min_render_period  = 1.0 / float(MAX_RENDER_FPS)

t0 = time.time()
last_event_t      = t0
last_render_t     = 0.0
last_sig          = None
last_status_t     = 0.0
last_status_frames = 0
last_status_polls = 0
frames            = 0
polls             = 0

try:
    while RUN_ENCODER_GUI and (time.time() - t0) < GUI_SECONDS:
        loop_t = time.time()
        polls += 1
        events = enc.poll(timestamp=loop_t - t0)
        if events:
            controller.handle_events(events)
            last_event_t = loop_t

        sig = _render_signature(state)
        if sig != last_sig and (loop_t - last_render_t) >= min_render_period:
            state.t = loop_t - t0
            frame = render_frame_800x480_compact_v2(state)
            backend.write_frame(frame, placement="manual",
                                offset_x=0, offset_y=0)
            last_sig = sig
            last_render_t = loop_t
            frames += 1

        if (loop_t - last_status_t) >= STATUS_INTERVAL_S:
            r = sampler.sample()
            dt = ((loop_t - last_status_t) if last_status_t > 0
                  else max(1e-3, loop_t - t0))
            render_fps = (frames - last_status_frames) / dt
            poll_hz = (polls - last_status_polls) / dt
            rss_mb = int(r.get("proc_rss_kb", 0)) // 1024
            mem_total_mb = int(r.get("mem_total_kb", 0)) // 1024
            mem_avail_mb = int(r.get("mem_avail_kb", 0)) // 1024
            mem_used_mb = (mem_total_mb - mem_avail_mb
                           if mem_total_mb else 0)
            mem_used_pct = (100.0 * mem_used_mb / mem_total_mb)                 if mem_total_mb else None
            idle_for = loop_t - last_event_t
            mode = "idle" if idle_for > IDLE_THRESHOLD_S else "active"
            msg = applier.last_apply_message or "-"
            if len(msg) > 28:
                msg = msg[:25] + "..."
            print(
                "t=%6.1fs mode=%-6s poll=%4.1fHz render=%4.1ffps "
                "sys_cpu=%s proc_cpu=%s mem=%4d/%4dMB(%s) rss=%4dMB "
                "temp=%s fx=%d knob=%d live=%s apply=%s last=%s"
                % (
                    loop_t - t0, mode, poll_hz, render_fps,
                    _fmt_pct(r.get("sys_cpu_pct")),
                    _fmt_pct(r.get("proc_cpu_pct")),
                    mem_used_mb, mem_total_mb, _fmt_pct(mem_used_pct),
                    rss_mb, _fmt_temp(r.get("temperature_c")),
                    state.selected_effect, state.selected_knob,
                    "ON" if state.live_apply else "off",
                    "OK" if applier.last_apply_ok else "ERR",
                    msg,
                )
            )
            last_status_t = loop_t
            last_status_frames = frames
            last_status_polls = polls

        idle_for = loop_t - last_event_t
        period = (poll_period_idle if idle_for > IDLE_THRESHOLD_S
                  else poll_period_active)
        elapsed = time.time() - loop_t
        if elapsed < period:
            time.sleep(period - elapsed)
except KeyboardInterrupt:
    pass
finally:
    backend.stop()
